# 04 · Build the `venues` table (SeatGeek + Census)

**Input:** `data/clean/events.csv` (from `02_events.ipynb`)
**Outputs:**
- `data/clean/venues.csv`: one row per venue (capacity, tier, SeatGeek score, county income)
- `data/clean/sg_event_scores.csv`: SeatGeek demand score per show, joined in the model step
- `data/manual/venue_capacity_manual.csv`: a short list of big venues SeatGeek has no capacity for, to fill in by hand

**Before running**, `.env` needs `SG_CLIENT_ID=...`. Census needs no key for this many calls (`CENSUS_API_KEY` is optional).

**Runtime:** about 10–15 minutes for ~480 venues. Answers are cached in `data/interim/`, so re-running only fetches what's missing.

In [1]:
import json
import math
import os
import re
import time
import unicodedata
from difflib import SequenceMatcher
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from dotenv import load_dotenv

load_dotenv(".env", override=True)
SG_ID = os.getenv("SG_CLIENT_ID")
CENSUS_KEY = os.getenv("CENSUS_API_KEY")       # optional
assert SG_ID, "Add SG_CLIENT_ID to .env"

USE_CENSUS = True
ACS_YEAR = 2023        # ACS 5-year release; try 2024 if it's available

CLEAN, INTERIM, MANUAL = Path("data/clean"), Path("data/interim"), Path("data/manual")
for p in (INTERIM, MANUAL):
    p.mkdir(parents=True, exist_ok=True)
SG_VENUE_CACHE = INTERIM / "seatgeek_venues.jsonl"
SG_EVENT_CACHE = INTERIM / "seatgeek_events.jsonl"
GEO_CACHE = INTERIM / "census_geocoder.jsonl"
SG = "https://api.seatgeek.com/2"
print("SeatGeek ID found:", bool(SG_ID))

SeatGeek ID found: True


## 1. One row per venue from events

In [2]:
events = pd.read_csv(CLEAN / "events.csv", low_memory=False, parse_dates=["date"])
venues = (events.groupby("venue_id")
                .agg(venue=("venue", "first"), city=("city", "first"), state=("state", "first"),
                     postal_code=("postal_code", "first"), lat=("lat", "first"), lon=("lon", "first"),
                     n_events=("event_id", "size"), median_price_min=("price_min", "median"))
                .reset_index()
                .sort_values("n_events", ascending=False))
print(f"{len(venues)} venues; top 10 hold {venues['n_events'].head(10).sum() / len(events):.0%} of events")
venues.head(10)

506 venues; top 10 hold 28% of events


,venue_id,venue,city,state,postal_code,lat,lon,n_events,median_price_min
95,rZ7HnEZ178J7g,Birdland Jazz Club,New York,NY,10036,40.759017,-73.989686,360,45.76
63,rZ7HnEZ174KrF,Birdland Theater,New York,NY,10036,40.759017,-73.989686,268,35.46
161,rZ7HnEZ17OeJf,Smalls Jazz Club,New York,NY,10014,40.734384,-74.002825,254,43.70
162,rZ7HnEZ17OePV,Mezzrow,New York,NY,10014,40.734396,-74.001888,243,43.70
471,rZ7HnEZaASU,Blue Note Jazz Club,New York,NY,10012,40.730940,-74.000650,163,38.10
482,rZ7HnEZadyf,3rd and Lindsley,Nashville,TN,37210,36.152076,-86.770085,145,27.70
402,rZ7HnEZ17qpvf,Blue Note Los Angeles,Los Angeles,CA,90028,34.097720,-118.328710,136,18.80
119,rZ7HnEZ178wGA,Belly Up,Solana Beach,CA,92075,32.990942,-117.270629,121,38.30
160,rZ7HnEZ17OeJ7,Jazzcultural,New York,NY,10036,40.760847,-73.989657,120,43.70
138,rZ7HnEZ17FaQ4,Whisky A Go Go,West Hollywood,CA,90069,34.090897,-118.385693,94,28.30


## 2. Helpers

In [3]:
def load_cache(path, key):
    if not path.exists():
        return {}
    with path.open(encoding="utf-8") as f:
        return {r[key]: r for r in (json.loads(l) for l in f if l.strip())}


def append_cache(path, row):
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(row) + "\n")


def get_json(url, params, pause=0.3, tries=4):
    for attempt in range(tries):
        try:
            r = requests.get(url, params=params, timeout=30)
        except requests.RequestException:
            time.sleep(2 * (attempt + 1))
            continue
        time.sleep(pause)
        if r.status_code in (429, 500, 502, 503):
            time.sleep(3 * (attempt + 1))
            continue
        if r.status_code in (401, 403):
            raise RuntimeError(f"Access refused ({r.status_code}): check the key. {r.text[:150]}")
        try:
            return r.json()
        except ValueError:
            return None
    return None


STOP = r"\b(the|presented by .*|at .*|music hall|theatre|theater|club|venue|live|ballroom|lounge|bar|tavern)\b"

def norm_venue(s):
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode().lower()
    s = re.sub(r"[^a-z0-9 ]+", " ", s.replace("&", " and "))
    core = re.sub(STOP, " ", s)
    return re.sub(r"\s+", " ", core if core.strip() else s).strip()


def name_sim(a, b):
    return SequenceMatcher(None, norm_venue(a), norm_venue(b)).ratio()


def km(lat1, lon1, lat2, lon2):
    if any(pd.isna(v) for v in (lat1, lon1, lat2, lon2)):
        return np.nan
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp, dl = p2 - p1, math.radians(lon2 - lon1)
    a = math.sin(dp / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
    return 6371 * 2 * math.asin(math.sqrt(a))

## 3. Match each venue to SeatGeek

Two searches per venue: by name within its state, then by location (everything within 1 mile). The best candidate is accepted if:
- the names are close (similarity 0.8+) and it's within 5 km, **or**
- the names are somewhat close (0.5+) and it's within 300 m (same building, different name).

In [4]:
def candidates(v):
    out = []
    js = get_json(f"{SG}/venues", {"client_id": SG_ID, "q": v.venue, "state": v.state, "per_page": 10})
    out += (js or {}).get("venues", [])
    if pd.notna(v.lat) and pd.notna(v.lon):
        js = get_json(f"{SG}/venues", {"client_id": SG_ID, "lat": v.lat, "lon": v.lon,
                                       "range": "1mi", "per_page": 50})
        out += (js or {}).get("venues", [])
    return out


def best_match(v, cands):
    best = None
    for c in cands:
        loc = c.get("location") or {}
        d = km(v.lat, v.lon, loc.get("lat"), loc.get("lon"))
        s = name_sim(v.venue, c.get("name", ""))
        ok = (s >= 0.8 and (pd.isna(d) or d <= 5)) or (s >= 0.5 and pd.notna(d) and d <= 0.3)
        if ok and (best is None or s > best["sim"]):
            best = {"sim": s, "dist_km": d, "venue": c}
    return best


cache = load_cache(SG_VENUE_CACHE, "venue_id")
todo = venues[~venues["venue_id"].isin(cache)]
print(f"SeatGeek venues: {len(cache)} cached, {len(todo)} to match")
for i, v in enumerate(todo.itertuples(), 1):
    m = best_match(v, candidates(v))
    c = m["venue"] if m else {}
    append_cache(SG_VENUE_CACHE, {
        "venue_id": v.venue_id, "matched": bool(m),
        "sg_venue_id": c.get("id"), "sg_name": c.get("name"),
        "sg_capacity": c.get("capacity") or None,          # SeatGeek uses 0 for unknown
        "sg_venue_score": c.get("score"), "sg_upcoming_events": c.get("num_upcoming_events"),
        "match_sim": m["sim"] if m else None, "match_km": m["dist_km"] if m else None})
    if i % 50 == 0:
        print(f"  {i}/{len(todo)}")

sgv = pd.DataFrame(load_cache(SG_VENUE_CACHE, "venue_id").values())
print(f"Matched: {sgv['matched'].mean():.0%} of venues | with capacity: {sgv['sg_capacity'].notna().mean():.0%}")

SeatGeek venues: 506 cached, 0 to match
Matched: 78% of venues | with capacity: 3%


In [5]:
# Spot-check matches: Ticketmaster name vs SeatGeek name
venues.merge(sgv, on="venue_id")[lambda d: d["matched"]] \
      .sort_values("match_sim")[["venue", "sg_name", "match_sim", "match_km", "sg_capacity"]].head(15)

,venue,sg_name,match_sim,match_km,sg_capacity
477,The Showroom at Wheeling Island,Wheeling Island Showroom,0.500000,0.272958,NaN
290,El Cid,El Cid - Los Angeles,0.500000,0.038268,NaN
140,The Fennec - Main Stage,"The Fennec - Sports, Live Music, Events",0.511628,0.005249,NaN
116,Lovin' Cup,Lovin' Cup Brews & Bistro,0.514286,0.082419,NaN
375,SANCTUARY (MA),Sanctuary Cultural Arts Center,0.523810,0.010521,NaN
73,SPACE,SPACE - Evanston,0.526316,0.012841,NaN
420,Rose Hall,The St Charles,0.526316,0.198769,NaN
55,Conduit,Conduit - Winter Park,0.538462,0.005006,NaN
273,NO DICE,No Dice - New Orleans,0.538462,0.053461,NaN
412,Denfeld Auditorium,Denfeld High School,0.540541,0.102544,NaN


## 4. SeatGeek demand score for each show
For every matched venue, pull its upcoming SeatGeek events and match ours by date.

In [6]:
ev_cache = load_cache(SG_EVENT_CACHE, "sg_venue_id")
matched_ids = sgv.loc[sgv["matched"], "sg_venue_id"].dropna().astype(int).unique()
todo = [i for i in matched_ids if i not in ev_cache and str(i) not in ev_cache]
print(f"SeatGeek events: {len(ev_cache)} venues cached, {len(todo)} to fetch")
for i, sid in enumerate(todo, 1):
    rows, page = [], 1
    while page <= 3:                                     # up to 300 upcoming shows per venue
        js = get_json(f"{SG}/events", {"client_id": SG_ID, "venue.id": int(sid), "per_page": 100, "page": page})
        evs = (js or {}).get("events", [])
        rows += [{"date": e.get("datetime_local", "")[:10], "score": e.get("score"), "title": e.get("title")}
                 for e in evs]
        if len(evs) < 100:
            break
        page += 1
    append_cache(SG_EVENT_CACHE, {"sg_venue_id": int(sid), "events": rows})
    if i % 50 == 0:
        print(f"  {i}/{len(todo)}")

sg_events = pd.DataFrame([{"sg_venue_id": int(r["sg_venue_id"]), **e}
                          for r in load_cache(SG_EVENT_CACHE, "sg_venue_id").values() for e in r["events"]])
if sg_events.empty:
    sg_events = pd.DataFrame(columns=["sg_venue_id", "date", "score", "title"])
sg_day = (sg_events.groupby(["sg_venue_id", "date"])
                   .agg(sg_event_score=("score", "max"), sg_event_title=("title", "first")).reset_index())

link = events[["event_id", "venue_id", "date"]].assign(date=lambda d: d["date"].dt.strftime("%Y-%m-%d"))
link = link.merge(sgv[["venue_id", "sg_venue_id"]].dropna(), on="venue_id")
link["sg_venue_id"] = link["sg_venue_id"].astype(int)
scores = link.merge(sg_day, on=["sg_venue_id", "date"], how="inner")[["event_id", "sg_event_score", "sg_event_title"]]
scores = scores.drop_duplicates("event_id")
scores.to_csv(CLEAN / "sg_event_scores.csv", index=False)
print(f"Saved sg_event_scores.csv: score for {len(scores)} of {len(events)} shows ({len(scores) / len(events):.0%})")

SeatGeek events: 384 venues cached, 0 to fetch
Saved sg_event_scores.csv: score for 785 of 6733 shows (12%)


## 5. Census: county income and population (optional)
The Census geocoder turns each venue's lat/lon into a county code; one ACS call then gives every county's median household income and population.

In [7]:
if USE_CENSUS:
    geo = load_cache(GEO_CACHE, "venue_id")
    todo = venues[~venues["venue_id"].isin(geo) & venues["lat"].notna()]
    print(f"Geocoder: {len(geo)} cached, {len(todo)} to fetch")
    for i, v in enumerate(todo.itertuples(), 1):
        js = get_json("https://geocoding.geo.census.gov/geocoder/geographies/coordinates",
                      {"x": v.lon, "y": v.lat, "benchmark": "Public_AR_Current",
                       "vintage": "Current_Current", "layers": "Counties", "format": "json"}, pause=0.2)
        counties = (((js or {}).get("result") or {}).get("geographies") or {}).get("Counties") or []
        append_cache(GEO_CACHE, {"venue_id": v.venue_id, "county_fips": counties[0]["GEOID"] if counties else None})
        if i % 100 == 0:
            print(f"  {i}/{len(todo)}")
    geo = pd.DataFrame(load_cache(GEO_CACHE, "venue_id").values())

    params = {"get": "NAME,B19013_001E,B01003_001E", "for": "county:*"}
    if CENSUS_KEY:
        params["key"] = CENSUS_KEY
    acs_rows = get_json(f"https://api.census.gov/data/{ACS_YEAR}/acs/acs5", params)
    acs = pd.DataFrame(acs_rows[1:], columns=acs_rows[0])
    acs["county_fips"] = acs["state"] + acs["county"]
    acs = acs.rename(columns={"NAME": "county_name", "B19013_001E": "median_income", "B01003_001E": "county_population"})
    acs[["median_income", "county_population"]] = acs[["median_income", "county_population"]].apply(pd.to_numeric, errors="coerce")
    acs.loc[acs["median_income"] < 0, "median_income"] = np.nan     # Census uses negative codes for missing
    census = geo.merge(acs[["county_fips", "county_name", "median_income", "county_population"]],
                       on="county_fips", how="left")
    print(f"Counties found: {census['county_fips'].notna().mean():.0%} | with income: {census['median_income'].notna().mean():.0%}")
else:
    census = pd.DataFrame(columns=["venue_id", "county_fips", "county_name", "median_income", "county_population"])

Geocoder: 506 cached, 0 to fetch
Counties found: 98% | with income: 98%


## 6. Hand-filled capacity for the busiest venues

The first run writes `data/manual/venue_capacity_manual.csv` listing the 40 busiest venues with no SeatGeek capacity. Look each up (venue website or Wikipedia), type the number in the `capacity` column, save, and re-run this notebook. Blank rows are ignored.

In [8]:
MANUAL_FILE = MANUAL / "venue_capacity_manual.csv"
base = venues.merge(sgv, on="venue_id", how="left")

if MANUAL_FILE.exists():
    manual = pd.read_csv(MANUAL_FILE)
    manual = manual[pd.to_numeric(manual["capacity"], errors="coerce").notna()][["venue_id", "capacity"]]
    print(f"Manual capacities loaded: {len(manual)}")
else:
    manual = pd.DataFrame(columns=["venue_id", "capacity"])
    template = base[base["sg_capacity"].isna()].head(40)[["venue_id", "venue", "city", "state", "n_events"]]
    template.assign(capacity="").to_csv(MANUAL_FILE, index=False)
    print(f"Wrote {MANUAL_FILE} with {len(template)} venues to fill in by hand")

Wrote data\manual\venue_capacity_manual.csv with 40 venues to fill in by hand


## 7. Combine and save

In [9]:
out = (base.merge(manual.rename(columns={"capacity": "manual_capacity"}), on="venue_id", how="left")
           .merge(census, on="venue_id", how="left"))
out["capacity"] = pd.to_numeric(out["manual_capacity"], errors="coerce").fillna(out["sg_capacity"])
out["capacity_source"] = np.select([out["manual_capacity"].notna(), out["sg_capacity"].notna()],
                                   ["manual", "seatgeek"], "missing")
out["venue_tier"] = pd.cut(out["capacity"], [0, 500, 2500, np.inf],
                           labels=["club (500 or less)", "theater (501-2,500)", "large (2,500+)"]).astype(str)
out.loc[out["capacity"].isna(), "venue_tier"] = "unknown"

cols = ["venue_id", "venue", "city", "state", "postal_code", "lat", "lon", "n_events", "median_price_min",
        "capacity", "capacity_source", "venue_tier", "sg_venue_id", "sg_name", "match_sim",
        "sg_venue_score", "sg_upcoming_events", "county_fips", "county_name", "median_income", "county_population"]
out = out[cols]
out.to_csv(CLEAN / "venues.csv", index=False)

print(f"Saved {CLEAN / 'venues.csv'}: {len(out)} venues")
print(f"  matched to SeatGeek: {out['sg_venue_id'].notna().mean():.0%}")
print(f"  capacity known:      {out['capacity'].notna().mean():.0%} of venues, "
      f"{events['venue_id'].isin(out.loc[out['capacity'].notna(), 'venue_id']).mean():.0%} of events")
print(f"  county income:       {out['median_income'].notna().mean():.0%}")
out["venue_tier"].value_counts().to_frame("venues")

Saved data\clean\venues.csv: 506 venues
  matched to SeatGeek: 78%
  capacity known:      3% of venues, 3% of events
  county income:       98%


,venues
venue_tier,
unknown,489
club (500 or less),7
"theater (501-2,500)",7
"large (2,500+)",3


In [10]:
# Large venues (2,500+) are outside our small/mid-venue scope: check them before the model step drops them
out[out["venue_tier"] == "large (2,500+)"][["venue", "city", "state", "capacity", "n_events"]]

,venue,city,state,capacity,n_events
178,The Masonic Lodge at Hollywood Forever,Los Angeles,CA,18118.0,6
344,Yost Theater,Santa Ana,CA,5500.0,1
389,Beacon Theatre,New York,NY,2894.0,1
